In [1]:
#Mount Drive and verify input file
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import os

skeletons_path = '/content/drive/MyDrive/HRC_Research/datasets/HRI30/extracted_skeletons/hri30_skeletons.npy'
labels_path = '/content/drive/MyDrive/HRC_Research/datasets/HRI30/extracted_skeletons/hri30_labels.npy'

# Create the new directory for the 70/10/20 split
new_base_dir = '/content/drive/MyDrive/HRC_Research/datasets/HRI30/HRI30_70_10_20'
os.makedirs(new_base_dir, exist_ok=True)
print(f"New directory created/verified: {new_base_dir}")

skeletons = np.load(skeletons_path)
labels = np.load(labels_path)

print("Skeletons shape:", skeletons.shape)   # Expected: (2940, 3, 150, 33)
print("Labels shape:", labels.shape)          # Expected: (2940,)
print("Unique classes:", np.unique(labels))   # Expected: 0–29
print("Samples per class:", np.bincount(labels))  # Expected: 98 for all 30 classes
print("dtype skeletons:", skeletons.dtype)
print("dtype labels:", labels.dtype)

Mounted at /content/drive
New directory created/verified: /content/drive/MyDrive/HRC_Research/datasets/HRI30/HRI30_70_10_20
Skeletons shape: (2940, 3, 150, 33)
Labels shape: (2940,)
Unique classes: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29]
Samples per class: [98 98 98 98 98 98 98 98 98 98 98 98 98 98 98 98 98 98 98 98 98 98 98 98
 98 98 98 98 98 98]
dtype skeletons: float32
dtype labels: int32


In [2]:
#Remap 33 MediaPipe joints → 25 NTU joints
direct_mapping = {
    3: 0,   4: 11,  5: 13,  6: 15,  7: 19,
    8: 12,  9: 14, 10: 16, 11: 20, 12: 23,
   13: 25, 14: 27, 15: 31, 16: 24, 17: 26,
   18: 28, 19: 32, 21: 17, 22: 21, 23: 18, 24: 22
}

N, C, T, _ = skeletons.shape  # (2940, 3, 150, 33)
ntu_skeletons = np.zeros((N, C, T, 25), dtype=np.float32)

# Direct mapping
for ntu_idx, mp_idx in direct_mapping.items():
    ntu_skeletons[:, :, :, ntu_idx] = skeletons[:, :, :, mp_idx]

# Virtual joints
ntu_skeletons[:, :, :, 0]  = (skeletons[:, :, :, 23] + skeletons[:, :, :, 24]) / 2.0  # Base Spine
ntu_skeletons[:, :, :, 2]  = (skeletons[:, :, :, 11] + skeletons[:, :, :, 12]) / 2.0  # Neck
ntu_skeletons[:, :, :, 1]  = (ntu_skeletons[:, :, :, 0] + ntu_skeletons[:, :, :, 2]) / 2.0  # Mid Spine
ntu_skeletons[:, :, :, 20] = ntu_skeletons[:, :, :, 1].copy()  # Spine 2

print("NTU skeleton shape:", ntu_skeletons.shape)  # Expected: (2940, 3, 150, 25)
print("No NaN values:", not np.isnan(ntu_skeletons).any())
print("Sample joint 0 range:", ntu_skeletons[:, :, :, 0].min(), "to", ntu_skeletons[:, :, :, 0].max())

NTU skeleton shape: (2940, 3, 150, 25)
No NaN values: True
Sample joint 0 range: -0.00012719864 to 0.9558594


In [3]:
#Add person axis and verify
ntu_expanded = np.expand_dims(ntu_skeletons, axis=4)
# Shape: (2940, 3, 150, 25, 1)

print("Expanded shape:", ntu_expanded.shape)  # Expected: (2940, 3, 150, 25, 1)

Expanded shape: (2940, 3, 150, 25, 1)


In [4]:
#Stratified 70/10/20 train/val/test split
from sklearn.model_selection import train_test_split

indices = np.arange(len(labels))

# Step 1: Split out the 20% test set (exact same 588 samples as before because random_state=42)
trainval_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

# Step 2: Split the remaining 80% into 70% train and 10% val (0.125 * 0.8 = 0.10)
train_idx, val_idx = train_test_split(
    trainval_idx,
    test_size=0.125,
    random_state=42,
    stratify=labels[trainval_idx]
)

X_train = ntu_expanded[train_idx]   # (2058, 3, 150, 25, 1)
X_val   = ntu_expanded[val_idx]    # (294, 3, 150, 25, 1)
X_test  = ntu_expanded[test_idx]   # (588, 3, 150, 25, 1)

y_train = labels[train_idx]
y_val   = labels[val_idx]
y_test  = labels[test_idx]

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)
print("y_test shape:", y_test.shape)
print("Train samples per class:", np.bincount(y_train))
print("Val samples per class:", np.bincount(y_val))
print("Test samples per class:", np.bincount(y_test))

X_train shape: (2058, 3, 150, 25, 1)
X_val shape: (294, 3, 150, 25, 1)
X_test shape: (588, 3, 150, 25, 1)
y_train shape: (2058,)
y_val shape: (294,)
y_test shape: (588,)
Train samples per class: [69 69 69 68 69 69 68 69 69 69 68 68 69 69 68 69 69 69 68 68 68 69 68 69
 69 68 68 69 69 68]
Val samples per class: [10  9  9 10 10  9 10  9 10 10 10 10 10  9 10 10 10 10 10 10 10  9 10 10
 10 10 10 10 10 10]
Test samples per class: [19 20 20 20 19 20 20 20 19 19 20 20 19 20 20 19 19 19 20 20 20 20 20 19
 19 20 20 19 19 20]


In [5]:
#Save ST-GCN format
import pickle

# Point to the new directory
stgcn_dir = '/content/drive/MyDrive/HRC_Research/datasets/HRI30/HRI30_70_10_20/stgcn_format'
os.makedirs(stgcn_dir, exist_ok=True)

# Save data arrays
np.save(os.path.join(stgcn_dir, 'train_data.npy'), X_train)
np.save(os.path.join(stgcn_dir, 'val_data.npy'), X_val)     # NEW
np.save(os.path.join(stgcn_dir, 'test_data.npy'), X_test)

# Save label pkl files
train_names = [f'v_{i}' for i in train_idx]
val_names   = [f'v_{i}' for i in val_idx]                   # NEW
test_names  = [f'v_{i}' for i in test_idx]

with open(os.path.join(stgcn_dir, 'train_label.pkl'), 'wb') as f:
    pickle.dump((train_names, list(y_train)), f)

with open(os.path.join(stgcn_dir, 'val_label.pkl'), 'wb') as f:       # NEW
    pickle.dump((val_names, list(y_val)), f)

with open(os.path.join(stgcn_dir, 'test_label.pkl'), 'wb') as f:
    pickle.dump((test_names, list(y_test)), f)

# Verify sizes
print("--- ST-GCN format files saved ---")
for fname in ['train_data.npy', 'val_data.npy', 'test_data.npy', 'train_label.pkl', 'val_label.pkl', 'test_label.pkl']:
    fpath = os.path.join(stgcn_dir, fname)
    size_mb = os.path.getsize(fpath) / 1024**2
    print(f"  {fname}: {size_mb:.1f} MB")

--- ST-GCN format files saved ---
  train_data.npy: 88.3 MB
  val_data.npy: 12.6 MB
  test_data.npy: 25.2 MB
  train_label.pkl: 0.1 MB
  val_label.pkl: 0.0 MB
  test_label.pkl: 0.0 MB


In [6]:
#Save CTR-GCN format
# Point to the new directory
ctrgcn_dir = '/content/drive/MyDrive/HRC_Research/datasets/HRI30/HRI30_70_10_20/ctrgcn_format'
os.makedirs(ctrgcn_dir, exist_ok=True)

npz_path = os.path.join(ctrgcn_dir, 'HRI30_CS.npz')

np.savez(
    npz_path,
    x_train=X_train,
    y_train=y_train,
    x_val=X_val,             # NEW
    y_val=y_val,             # NEW
    x_test=X_test,
    y_test=y_test
)

# Verify
size_mb = os.path.getsize(npz_path) / 1024**2
print(f"HRI30_CS.npz: {size_mb:.1f} MB")

# Reload and check keys and shapes
loaded = np.load(npz_path)
for k in loaded.files:
    print(f"  {k}: shape {loaded[k].shape}, dtype {loaded[k].dtype}")

HRI30_CS.npz: 126.2 MB
  x_train: shape (2058, 3, 150, 25, 1), dtype float32
  y_train: shape (2058,), dtype int32
  x_val: shape (294, 3, 150, 25, 1), dtype float32
  y_val: shape (294,), dtype int32
  x_test: shape (588, 3, 150, 25, 1), dtype float32
  y_test: shape (588,), dtype int32


In [7]:
#Final verification summary
print("=" * 50)
print("PHASE 2.2 COMPLETE — VERIFICATION SUMMARY")
print("=" * 50)

stgcn_dir = '/content/drive/MyDrive/HRC_Research/datasets/HRI30/HRI30_70_10_20/stgcn_format'
ctrgcn_dir = '/content/drive/MyDrive/HRC_Research/datasets/HRI30/HRI30_70_10_20/ctrgcn_format'

print("\nST-GCN format files:")
for fname in ['train_data.npy', 'val_data.npy', 'test_data.npy', 'train_label.pkl', 'val_label.pkl', 'test_label.pkl']:
    fpath = os.path.join(stgcn_dir, fname)
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / 1024**2
        print(f"  {fname}: {size_mb:.1f} MB")
    else:
        print(f"  MISSING: {fname}")

print("\nCTR-GCN format file:")
npz_path = os.path.join(ctrgcn_dir, 'HRI30_CS.npz')
if os.path.exists(npz_path):
    size_mb = os.path.getsize(npz_path) / 1024**2
    print(f"  HRI30_CS.npz: {size_mb:.1f} MB")
else:
    print("  MISSING: HRI30_CS.npz")

print("\nAll 70/10/20 outputs saved to Drive.")
print("Phase 2.2 done. Ready for Phase 2.3 (ST-GCN fine-tuning).")

PHASE 2.2 COMPLETE — VERIFICATION SUMMARY

ST-GCN format files:
  train_data.npy: 88.3 MB
  val_data.npy: 12.6 MB
  test_data.npy: 25.2 MB
  train_label.pkl: 0.1 MB
  val_label.pkl: 0.0 MB
  test_label.pkl: 0.0 MB

CTR-GCN format file:
  HRI30_CS.npz: 126.2 MB

All 70/10/20 outputs saved to Drive.
Phase 2.2 done. Ready for Phase 2.3 (ST-GCN fine-tuning).
